In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

# Set the environment variable for Google credentials
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "/home/karlchee/assignment_Module2/academic-era-488315-j5-52715453be94.json"

# Create engine without passing credentials in connect_args
engine = create_engine("bigquery://academic-era-488315-j5")

In [14]:
# Extract the monthly revenue with month start date
query = """
WITH monthly_stamped_revenue AS (
    SELECT
        a.order_purchase_date,
        a.payment_value,
        b.year, b.month
    FROM olist_data_warehouse.order_items as a
    JOIN olist_data_warehouse.date as b
    ON a.order_purchase_date = b.full_date
)

SELECT
    DATE(CONCAT(CAST(year AS STRING), '-', LPAD(CAST(month AS STRING), 2, '0'), '-01')) as month_start,
    SUM(payment_value) as monthly_revenue
FROM monthly_stamped_revenue
GROUP BY month_start
ORDER BY month_start
"""

df_revenue = pd.read_sql(query, engine)
print(df_revenue.head(10))

  month_start  monthly_revenue
0  2016-09-01     3.616211e+03
1  2016-10-01     1.457603e+06
2  2016-12-01     9.504800e+01
3  2017-01-01     2.811007e+06
4  2017-02-01     8.417254e+06
5  2017-03-01     1.028118e+07
6  2017-04-01     1.152930e+07
7  2017-05-01     1.558488e+07
8  2017-06-01     1.258609e+07
9  2017-07-01     1.300447e+07


In [15]:
# Plot the revenue trend as a time series
import plotly.express as px
fig = px.line(df_revenue, x='month_start', y='monthly_revenue', markers=True,
              title='Monthly Revenue Trend Over Time')
fig.update_layout(xaxis_title='Month', yaxis_title='Revenue')
fig.show()
